# 5. The standard library and model validation

A curated subset of the official OMG model library ships with the
package: all 21 Systems Library files, the core Quantities-and-Units
files, and a shim for the KerML kernel names. It loads in milliseconds
from a prebuilt JSON serialization (the same lossless schema as
`to_json`, with no pickles).

**You will learn how to:**

- load the vendored standard library and resolve names through it;
- attach the library to a user model (`add_standard_library`);
- validate a model and read each diagnostic (`validate`).

**Prerequisites:** tutorial 1. The library and the validator are core
features, so no extras are needed.

The first two cells load the library, list its packages, and resolve
names through `public import` re-exports and aliases.

In [ ]:
import longeron

library = longeron.standard_library_model()
print(len(library.members), "packages:")
print(", ".join(sorted(m.name for m in library.members)[:14]), "...")

In [ ]:
interp = longeron.Interpreter(library)
for name in ("Parts::Part", "Actions::Action", "ScalarValues::Real", "ISQ::mass", "SI::kg"):
    print(f"{name:20s} -> {interp.resolve(name).qualified_name}")
# ISQ::mass resolves through a `public import` re-export;
# SI::kg resolves through an alias.

## User models against the library

`add_standard_library(model)` attaches the library, so `Parts::Part`
and `ScalarValues::Real` resolve, and `istype` checks work against
library definitions.

In [ ]:
model = longeron.loads("""
package App {
    private import ScalarValues::*;
    private import Parts::*;
    part def Robot :> Part {
        attribute mass : Real = 12.0;
    }
}
""")
longeron.add_standard_library(model)

interp = longeron.Interpreter(model)
robot = interp.instantiate("App::Robot")
print("mass:", robot.slots["mass"])
print("robot istype Part:", interp.evaluate("r istype Part", context="App", r=robot))

## Validation: `longeron.validate()` / `longeron lint`

Structural problems are errors, and unresolved references are warnings.
Validation is stdlib-aware: names resolve against the vendored standard
library, so a bare `Real` passes without any import while a typo like
`Reall` warns. Plain definitions also get their *implied*
specializations: a `part def` is implicitly a `Parts::Part`, and an
`action def` an `Actions::Action`, which is how `start` and `done`
resolve. Opt out with `validate(model, stdlib=False)` or
`longeron lint --no-stdlib`.

The `Buggy` package below plants several problems. Each prints as one
diagnostic: `severity[code] element: message`. The `Clean` package
validates silently.

In [ ]:
buggy = longeron.loads("""
package Buggy {
    part def X;
    part def X;                          // duplicate name
    part def A :> B;
    part def B :> A;                     // specialization cycle
    part ghost : NoSuchType;             // dangling reference
    part def V {
        attribute a : Real = 1.0;
        attribute b : Real = aa + 1.0;   // typo in expression
    }
    state def S {
        state on;                        // no entry transition
        transition first on accept go then off;   // unknown target
    }
}
""")
for diagnostic in longeron.validate(buggy):
    print(diagnostic)

In [ ]:
clean = longeron.loads("""
package Clean {
    private import Parts::*;
    part def Robot :> Part { attribute mass : Real = 1.0; }
}
""")
longeron.add_standard_library(clean)
print("diagnostics:", [d for d in longeron.validate(clean) if "Clean" in d.element] or "none")